# Google Colab Runner — Modelo Gate (cardíaco vs. pulmonar)

Este notebook permite entrenar y evaluar el modelo gate del proyecto usando la GPU de Google Colab, reutilizando los scripts existentes en `desarrollo/src/gate_model/` sin modificarlos (salvo detalles internos de rendimiento).

**Requisito previo**: la carpeta `desarrollo/` del proyecto debe estar sincronizada en tu Google Drive (incluyendo `data/unified_v2/` y `data/gate/windows/`, ya generados localmente).

**Antes de correr**: activá el runtime con GPU en `Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware → GPU`.

Cada celda imprime qué etapa está ejecutando para poder seguir el progreso.

**Nota sobre rendimiento**: leer directamente desde Google Drive montado (miles de archivos `.wav` chicos) es muy lento por la latencia de red de cada apertura de archivo — mucho más lento que la GPU en sí. Por eso, antes de entrenar, este notebook copia las ventanas de audio (`data/gate/`) a disco local de la VM de Colab en paralelo.

In [16]:
print("=== Montando Google Drive ===")
from google.colab import drive
drive.mount('/content/drive')

=== Montando Google Drive ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
print("=== Configurando ruta del proyecto ===")

# Editá esta ruta si tu carpeta "desarrollo" está en otra ubicación de tu Drive.
PROJECT_PATH = "/content/drive/MyDrive/CEIA/Desarrollo_Proyecto_final_CEIA/"

import os
assert os.path.isdir(PROJECT_PATH), f"No se encontró la carpeta: {PROJECT_PATH}"
os.chdir(os.path.join(PROJECT_PATH, "src"))
print(f"✓ Directorio de trabajo: {os.getcwd()}")

=== Configurando ruta del proyecto ===
✓ Directorio de trabajo: /content/drive/Othercomputers/santi_pc/Desarrollo_Proyecto_final_CEIA/src


In [18]:
print("=== Instalando dependencias ===")
!pip install -q -r ../requirements.txt

=== Instalando dependencias ===


In [19]:
print("=== Verificando GPU ===")
import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))

=== Verificando GPU ===
CUDA disponible: True
Dispositivo: Tesla T4


## Copiar las ventanas de audio a disco local

`train.py`/`evaluate.py` leen miles de archivos `.wav` individuales por época (una vez por cada ventana, en cada época). Hacerlo contra Drive montado es extremadamente lento porque cada apertura de archivo tiene latencia de red. Copiamos `data/gate/` una sola vez a `/content/gate_data` (disco local de la VM) usando varios hilos en paralelo para esconder esa latencia, y le indicamos a los scripts que usen esa copia local mediante la variable de entorno `GATE_DATA_DIR` (soportada en `common/paths.py`; si no está definida, el comportamiento es exactamente el mismo de siempre).

Esta copia se pierde si el runtime de Colab se reinicia o desconecta, así que hay que volver a correr esta celda en ese caso.

In [20]:
print("=== Copiando data/gate a disco local de la VM ===")
import os
import shutil
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

SRC_GATE_DIR = Path(PROJECT_PATH) / "data" / "gate"
LOCAL_GATE_DIR = Path("/content/gate_data")

files = [p for p in SRC_GATE_DIR.rglob("*") if p.is_file()]
print(f"Archivos a copiar: {len(files)}")

def _copy_one(src: Path) -> None:
    dst = LOCAL_GATE_DIR / src.relative_to(SRC_GATE_DIR)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

start = time.time()
with ThreadPoolExecutor(max_workers=32) as pool:
    list(pool.map(_copy_one, files))
print(f"✓ Copia completa en {time.time() - start:.1f}s → {LOCAL_GATE_DIR}")

os.environ["GATE_DATA_DIR"] = str(LOCAL_GATE_DIR)
print(f"✓ GATE_DATA_DIR = {os.environ['GATE_DATA_DIR']}")

=== Copiando data/gate a disco local de la VM ===
Archivos a copiar: 35820
✓ Copia completa en 908.3s → /content/gate_data
✓ GATE_DATA_DIR = /content/gate_data


In [21]:
print("=== Entrenando modelo gate ===")
!python gate_model/train.py

=== Entrenando modelo gate ===
Dispositivo: cuda
Ventanas — train: 24442, val: 5584
Balance de train — positivos: 19031, negativos: 5411, pos_weight: 0.284
[01/15] train_loss=0.0234 train_acc=0.9872 train_f1=0.9917 | val_loss=0.0118 val_acc=0.9928 val_f1=0.9951
  ✓ Nuevo mejor checkpoint (val_f1=0.9951)
[02/15] train_loss=0.0074 train_acc=0.9950 train_f1=0.9968 | val_loss=0.0178 val_acc=0.9830 val_f1=0.9884
Traceback (most recent call last):
  File "/content/drive/Othercomputers/santi_pc/Desarrollo_Proyecto_final_CEIA/src/gate_model/train.py", line 154, in <module>
    train()
    ~~~~~^^
  File "/content/drive/Othercomputers/santi_pc/Desarrollo_Proyecto_final_CEIA/src/gate_model/train.py", line 106, in train
    train_loss, train_acc, train_f1 = _run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                                      ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/Othercomputers/santi_pc/Desarrollo

In [22]:
print("=== Evaluando modelo gate (test) ===")
!python gate_model/evaluate.py

=== Evaluando modelo gate (test) ===
Ventanas de test: 5192
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 1023, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py", line 32, in <module>
    samplerate = lazy.load("samplerate")
  File "/usr/local/lib/python3.13/dist-packages/lazy_loader/__init__.py", line 227, in load
    parent = inspect.stack()[1]
             ~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/inspect.py", line 1768, in stack
    return getouterframes(sys._getframe(1), context)
  File "/usr/lib/python3.13/inspect.py", line 1743, in getouterframes
    traceback_info = getframeinfo(frame, context)
  File "/usr/lib/python3.13/inspect.py", line 1705, in getframeinfo
    lines, lnum 

In [ ]:
print("=== Evaluando modelo gate (validación externa HLS-CMDS) ===")
!python gate_model/evaluate_external_hls.py

=== Evaluando modelo gate (validación externa HLS-CMDS) ===


## Listo

Los checkpoints (`gate_model/checkpoints/best.pt`) y los reportes (`reports/gate/`) se escriben directamente en Drive (son pocos archivos, no hace falta copiarlos localmente), así que también estarán disponibles desde tu máquina local por la sincronización.